# Bangladesh Urban Center — Clean Sentinel-2 Preprocessing

This notebook replaces the duplicated/debug cells in `01.ipynb` with one reproducible workflow.

**Workflow:** AOI → STAC search → Bangladesh-intersecting scenes → best scenes per MGRS tile → coverage completion → resilient StackSTAC → SCL cloud mask → median composite → NDVI/NDBI/MNDWI → GeoTIFF export.

The notebook does **not** reinstall packages inside the running Conda environment. Install dependencies once in the `geo` environment, then restart the kernel.

In [2]:
# 1. CONFIGURATION
from pathlib import Path
import os

# Change only this path if your repository is elsewhere.
PROJECT_DIR = Path(r"E:\Geospatial\Urban Center\Urban-Center")
AOI_PATH = PROJECT_DIR / "bgd_admin_boundaries.shp" / "bgd_admin0.shp"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_RANGE = "2025-01-01/2025-03-31"
MAX_CLOUD = 10
N_BEST_PER_TILE = 5
TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

# Make remote COG reads more tolerant of temporary network/S3 errors.
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["GDAL_HTTP_RETRY_CODES"] = "ALL"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

print("AOI:", AOI_PATH)
print("Outputs:", OUTPUT_DIR)

AOI: E:\Geospatial\Urban Center\Urban-Center\bgd_admin_boundaries.shp\bgd_admin0.shp
Outputs: E:\Geospatial\Urban Center\Urban-Center\outputs


In [3]:
# 2. IMPORTS
from collections import defaultdict

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.errors import RasterioIOError
from shapely.geometry import shape, mapping
from shapely.ops import unary_union
from pystac_client import Client
import stackstac
import rioxarray  # registers the .rio accessor

print("Rasterio:", rasterio.__version__)
print("GDAL:", rasterio.__gdal_version__)

Rasterio: 1.4.4
GDAL: 3.10.3


In [4]:
# 3. LOAD BANGLADESH AOI
if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI shapefile not found: {AOI_PATH}\n"
        "Edit PROJECT_DIR/AOI_PATH in Cell 1."
    )

bd = gpd.read_file(AOI_PATH)
if bd.crs is None:
    raise ValueError("AOI has no CRS information.")

bd = bd.to_crs(4326)
aoi = bd[["geometry"]].dissolve().reset_index(drop=True)
bd_geom = aoi.geometry.iloc[0]
west, south, east, north = aoi.total_bounds

print("CRS:", aoi.crs)
print("Bounds:", (west, south, east, north))
print("AOI valid:", bd_geom.is_valid)

CRS: EPSG:4326
Bounds: (np.float64(88.00816912400006), np.float64(20.590608254000188), np.float64(92.68005782300008), np.float64(26.634548266000024))
AOI valid: True


In [5]:
# ============================================================
# SENTINEL-2 SEARCH - FIXED VERSION
# Uses bounding box instead of sending full Bangladesh geometry
# ============================================================

from pystac_client import Client
from shapely.geometry import shape

catalog = Client.open(
    "https://earth-search.aws.element84.com/v1"
)

# Bangladesh bounding box
bbox = list(bd_geom.bounds)

print("Searching Sentinel-2...")
print("BBox:", bbox)
print("Date range:", DATE_RANGE)
print("Max cloud:", MAX_CLOUD)

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=bbox,
    datetime=DATE_RANGE,
    query={
        "eo:cloud_cover": {
            "lt": MAX_CLOUD
        }
    },
    max_items=None
)

items = list(search.items())

print("Raw search results:", len(items))

# ------------------------------------------------------------
# Keep only scenes actually intersecting Bangladesh
# Because bbox can also return scenes slightly outside AOI
# ------------------------------------------------------------

items_bd = []

for item in items:
    try:
        scene_geom = shape(item.geometry)

        if scene_geom.intersects(bd_geom):
            items_bd.append(item)

    except Exception as e:
        print(
            f"Geometry check skipped: {item.id} | "
            f"{type(e).__name__}"
        )

items = items_bd

print("Scenes intersecting Bangladesh:", len(items))

if len(items) == 0:
    raise RuntimeError(
        "No Sentinel-2 scenes found. "
        "Check DATE_RANGE and MAX_CLOUD."
    )

# ------------------------------------------------------------
# Basic information
# ------------------------------------------------------------

dates = sorted(
    set(
        item.datetime.strftime("%Y-%m-%d")
        for item in items
        if item.datetime is not None
    )
)

tiles = sorted(
    set(
        item.properties.get("grid:code")
        or item.properties.get("mgrs:utm_zone", "")
        for item in items
    )
)

print("Unique acquisition dates:", len(dates))
print("First dates:", dates[:10])

print("\nSearch completed successfully.")

Searching Sentinel-2...
BBox: [88.00816912400006, 20.590608254000188, 92.68005782300008, 26.634548266000024]
Date range: 2025-01-01/2025-03-31
Max cloud: 10
Raw search results: 984
Scenes intersecting Bangladesh: 634
Unique acquisition dates: 51
First dates: ['2025-01-02', '2025-01-03', '2025-01-05', '2025-01-07', '2025-01-10', '2025-01-12', '2025-01-13', '2025-01-15', '2025-01-17', '2025-01-18']

Search completed successfully.


In [6]:
# 5. SELECT LOW-CLOUD SCENES AND GUARANTEE AOI FOOTPRINT COVERAGE
# Tile ID is the second underscore-delimited token, e.g. T45QYG.
items_by_tile = defaultdict(list)
for item in items_bd:
    tile_id = item.id.split("_")[1]
    items_by_tile[tile_id].append(item)

selected_items = []
for tile_id, tile_items in items_by_tile.items():
    tile_items = sorted(
        tile_items,
        key=lambda x: x.properties.get("eo:cloud_cover", 100),
    )
    selected_items.extend(tile_items[:N_BEST_PER_TILE])

# Add extra low-cloud scenes only where the initial selection leaves a footprint gap.
def footprint_union(stac_items):
    return unary_union([shape(item.geometry) for item in stac_items])

selected_union = footprint_union(selected_items)
missing_geom = bd_geom.difference(selected_union)

selected_ids = {item.id for item in selected_items}
remaining_items = sorted(
    [item for item in items_bd if item.id not in selected_ids],
    key=lambda x: x.properties.get("eo:cloud_cover", 100),
)

added_items = []
for item in remaining_items:
    if missing_geom.is_empty:
        break
    scene_geom = shape(item.geometry)
    if not missing_geom.intersection(scene_geom).is_empty:
        selected_items.append(item)
        added_items.append(item)
        selected_union = selected_union.union(scene_geom)
        missing_geom = bd_geom.difference(selected_union)

# Area-based coverage check in an equal-area CRS.
covered = bd_geom.intersection(selected_union)
coverage_gdf = gpd.GeoDataFrame(
    geometry=[bd_geom, covered], crs="EPSG:4326"
).to_crs(TARGET_EPSG)
coverage_pct = 100 * coverage_gdf.geometry.iloc[1].area / coverage_gdf.geometry.iloc[0].area

print("Unique MGRS tiles:", len(items_by_tile))
print("Selected scenes:", len(selected_items))
print("Extra scenes added for coverage:", len(added_items))
print(f"Bangladesh footprint coverage: {coverage_pct:.6f}%")

Unique MGRS tiles: 36
Selected scenes: 190
Extra scenes added for coverage: 10
Bangladesh footprint coverage: 100.000000%


In [7]:
# 6. BUILD ONE RESILIENT STACK — DO NOT OVERWRITE IT LATER
REQUIRED_ASSETS = ["green", "red", "nir", "swir16", "scl"]

# Remove items that do not advertise every required asset.
selected_items_clean = [
    item for item in selected_items
    if all(asset in item.assets for asset in REQUIRED_ASSETS)
]

missing_asset_count = len(selected_items) - len(selected_items_clean)
print("Scenes with all required assets:", len(selected_items_clean))
print("Dropped for missing asset metadata:", missing_asset_count)

if not selected_items_clean:
    raise RuntimeError("No selected scene contains all required Sentinel-2 assets.")

# Key fix for your last crash:
# StackSTAC can convert Rasterio read failures to NoData instead of stopping
# the entire national composite. Other dates for the same tile can still
# contribute to the median composite.
sentinel = stackstac.stack(
    selected_items_clean,
    assets=REQUIRED_ASSETS,
    bounds_latlon=[west, south, east, north],
    epsg=TARGET_EPSG,
    resolution=RESOLUTION_M,
    chunksize=CHUNK_SIZE,
    dtype="float32",
    fill_value=np.nan,

    # IMPORTANT FIX
    rescale=False,

    errors_as_nodata=(
        RasterioIOError(r".*"),
    ),
)

print(sentinel)
print("Virtual stack shape:", sentinel.shape)

Scenes with all required assets: 190
Dropped for missing asset metadata: 0


ValueError: The fill_value nan is incompatible with the output dtype float32. Either use `dtype='float64'`, or pick a different `fill_value`.

In [ ]:
# 7. CLOUD/SHADOW MASK + TEMPORAL MEDIAN
# Sentinel-2 Scene Classification Layer (SCL):
# mask NoData, saturated/defective, cloud shadow, medium/high cloud,
# cirrus and snow/ice. Keep vegetation, bare, water, etc.
scl = sentinel.sel(band="scl")
invalid_scl = [0, 1, 3, 8, 9, 10, 11]
valid = ~scl.isin(invalid_scl)

bands = {
    "green": sentinel.sel(band="green").where(valid),
    "red": sentinel.sel(band="red").where(valid),
    "nir": sentinel.sel(band="nir").where(valid),
    "swir16": sentinel.sel(band="swir16").where(valid),
}

# Median is robust to residual outliers and ignores NoData from failed reads.
green_med = bands["green"].median("time", skipna=True)
red_med = bands["red"].median("time", skipna=True)
nir_med = bands["nir"].median("time", skipna=True)
swir_med = bands["swir16"].median("time", skipna=True)

print("Median composites prepared lazily.")

In [ ]:
# 8. SPECTRAL INDICES
EPS = np.float32(1e-6)

def safe_nd(a, b):
    denom = a + b
    return ((a - b) / denom.where(np.abs(denom) > EPS)).clip(-1, 1)

ndvi = safe_nd(nir_med, red_med).rename("NDVI")
ndbi = safe_nd(swir_med, nir_med).rename("NDBI")
mndwi = safe_nd(green_med, swir_med).rename("MNDWI")

print("Prepared: NDVI, NDBI, MNDWI")

In [ ]:
# 9. QUICK TEST — SMALL WINDOW FIRST
# This triggers real remote reads but only for a small subset.
# If this succeeds, the corrected pipeline is working before you run the full export.
test = ndbi.isel(
    x=slice(1500, min(1800, ndbi.sizes["x"])),
    y=slice(2500, min(2800, ndbi.sizes["y"])),
).compute()

print(test)
print("Finite test pixels:", int(np.isfinite(test.values).sum()))

In [ ]:
# 10. OPTIONAL PREVIEW
preview = ndbi.coarsen(x=8, y=8, boundary="trim").mean().compute()

fig, ax = plt.subplots(figsize=(9, 10))
preview.plot(ax=ax, cmap="RdYlBu_r", vmin=-0.5, vmax=0.5)
ax.set_title("Bangladesh NDBI Preview (2025 Q1)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# 11. EXPORT INDICES AS GEOTIFF
# The first full export can take time because it triggers remote reads for Bangladesh.
def prepare_for_rio(da):
    return da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False).rio.write_crs(
        f"EPSG:{TARGET_EPSG}", inplace=False
    )

outputs = {
    "NDVI_2025_Q1_100m.tif": ndvi,
    "NDBI_2025_Q1_100m.tif": ndbi,
    "MNDWI_2025_Q1_100m.tif": mndwi,
}

for filename, da in outputs.items():
    path = OUTPUT_DIR / filename
    print("Writing:", path)
    prepare_for_rio(da).rio.to_raster(
        path,
        dtype="float32",
        compress="DEFLATE",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

print("Done.")

## Why the old last cell failed

1. The scene-check loop tested only `swir16`, but the crash happened while reading `nir` (`B08.tif`), so the test could report every scene as good while a different required band was unreadable.
2. A manually cleaned list removed one specific scene, but the failing scene later was a different ID; hard-coding one bad scene is therefore brittle.
3. After creating a clean stack, the notebook later created `sentinel` again from the original `selected_items`, overwriting the clean stack.
4. Repeated stack/composite cells made it difficult to know which variable version was active.
5. Installing/upgrading geospatial packages from inside an already-running Conda notebook can destabilize Rasterio/PyProj/PROJ; keep environment installation outside the analysis notebook.